# Predicting tomorrow's air quality in Ohio: the simplest version (Altair charts + XGBoost)

**The idea:** look at **today** (air quality, weather, time of year) and guess **tomorrow's** air quality. Then check how good the guesses are on days the computer has never seen.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. Everything runs in about a minute.

**Words you will see**
- **AQI** (Air Quality Index): higher means worse air. 0-50 is good, and above 100 is unhealthy for sensitive people.
- **Clue**: a fact we give the computer to help it guess, for example today's AQI.
- **Baseline**: the lazy guess "tomorrow will be the same as today". A model is only useful if it beats this.
- **MAE**: how many AQI points the guess is wrong by, on average. Smaller is better.
- **RMSE**: like MAE, but big mistakes count extra. Smaller is better.
- **XGBoost**: a model made of many small decision trees. Each new tree tries to fix the mistakes of the trees before it.
- **Altair**: the Python library that draws our charts. Colab normally has it already, and XGBoost too.

**Note:** this is a shortened version of the poster analysis (12 clues and one model). The poster's best models are a tuned random forest and a similar boosting model, so XGBoost's numbers should be close to the poster's, but not identical. The "tomorrow = today" row is exactly the same.

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row of the file is one county on one day: its AQI and the weather.

In [ ]:
import pandas as pd
import altair as alt
from google.colab import files

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first

print(len(df), "rows |", df.County.nunique(), "counties |", df.Date.min().date(), "to", df.Date.max().date())
df[["Date", "County", "AQI"]].head()                  # the first 5 rows

## Step 2. Look at the data: when is the air worst?

We draw the charts with **Altair**. An Altair chart has three parts:
1. `alt.Chart(table)`: which table to draw.
2. `.mark_boxplot()` (or `.mark_bar()`, `.mark_line()`): which shape to draw.
3. `.encode(x=..., y=...)`: which column goes where. The letter after the colon says what kind of column it is: `Q` = a number, `N` = a name, `T` = a date.

First we put every day into a quarter of the year (Q1 = Jan-Mar, Q2 = Apr-Jun, Q3 = Jul-Sep, Q4 = Oct-Dec). A **boxplot** then shows how the AQI is spread out in each quarter: the box holds the middle half of the days, the line inside the box is the median (the middle day), and the dots are unusual days.

In [ ]:
df["Quarter"] = "Q" + df.Date.dt.quarter.astype(str)      # pandas knows each date's quarter: 1, 2, 3 or 4
print(df.groupby("Quarter").AQI.median())                 # the middle AQI of each quarter

alt.data_transformers.disable_max_rows()      # Altair stops at 5,000 rows by default. This chart uses all 49,000 days.

alt.Chart(df[["Quarter", "AQI"]]).mark_boxplot().encode(
    x=alt.X("Quarter:N", axis=alt.Axis(labelAngle=0)),     # labelAngle=0 keeps the names horizontal
    y=alt.Y("AQI:Q", title="Daily AQI"),
).properties(title="Daily AQI by quarter", width=450, height=300)

**What this means**
- Q2 and Q3 (April to September) are the worst quarters: their middle values are 45 and 44, against 41 in Q1 and 38 in Q4.
- The dots above each box are the rare very bad days. They are the days we most want to predict. In Step 7 we will see how the model does on them.

## Step 3. Make the clues

We want to guess **tomorrow's AQI** in each county. So for every county and day we prepare the **answer** (`tomorrow`) and the **clues** the computer may use:

| Clue | What it is |
|---|---|
| `aqi_today` | today's AQI in the county |
| `aqi_yesterday` | yesterday's AQI |
| `aqi_7_day_average` | the average AQI of the last 7 days |
| `month` | the month number (1 = January ... 12 = December) |
| `county_average` | the county's usual AQI in 2022-2024 |
| 7 weather columns | today's temperature, rain, wind, humidity and sunshine |

In [ ]:
each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before
df["month"] = df.Date.dt.month

# The county's usual AQI. We use 2022-2024 only, so the test days (2025 and later) stay unseen.
usual = df[df.Date < "2025-01-01"].groupby("County").AQI.mean()
df["county_average"] = df.County.map(usual)

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]
clues = ["aqi_today", "aqi_yesterday", "aqi_7_day_average", "month", "county_average"] + weather

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
df = df[next_row_is_tomorrow].dropna(subset=clues + ["tomorrow"])
print("Days we can use:", len(df))

## Step 4. Learn from the past, test on the future

We **train** (let the computer learn) on the days before 2025, and we **test** (check its guesses) on 2025 and later. We never mix them, because a real forecast cannot see the future.

In [ ]:
train = df[df.Date < "2025-01-01"]        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"]        # 2025 to May 2026: we check its guesses on these days

X_train, y_train = train[clues], train.tomorrow     # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

## Step 5. Train XGBoost and compare it with the baseline

**XGBoost** builds small decision trees one after another. The first tree makes a rough guess, the second tree tries to fix the mistakes of the first, the third fixes what is still wrong, and so on. Many simple models working together is called an *ensemble*.

We compare two ways of guessing:
- **Baseline:** tomorrow = today. No learning at all.
- **XGBoost:** 300 small trees. We did not tune its settings: they are common starting values.

In [ ]:
from xgboost import XGBRegressor                 # if this says "No module named xgboost", run a new cell with: !pip install xgboost

model = XGBRegressor(n_estimators=300,           # 300 small trees, built one after another
                     learning_rate=0.05,         # each new tree fixes only a small part of the mistakes
                     max_depth=4,                # each tree asks at most 4 questions in a row
                     random_state=42)            # a fixed number, so you get the same result every run
model.fit(X_train, y_train)                      # build the trees from the training days
print("XGBoost has learned from", len(X_train), "training days.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

guesses = {"Baseline": X_test.aqi_today,           # the lazy guess: tomorrow = today
           "XGBoost": model.predict(X_test)}       # XGBoost's guess for every test day

errors = {}
for name, guess in guesses.items():                # grade each guess against the real AQI
    errors[name] = {"MAE": mean_absolute_error(y_test, guess),
                    "RMSE": mean_squared_error(y_test, guess) ** 0.5}

table = pd.DataFrame(errors).T                     # one row per model
print(table.round(2))
print(f"XGBoost's RMSE is {1 - table.RMSE['XGBoost'] / table.RMSE['Baseline']:.0%} lower than the baseline.")

In [ ]:
alt.Chart(table.rename_axis("Model").reset_index()).mark_bar().encode(
    x=alt.X("MAE:Q", title="Average miss in AQI points (smaller is better)"),
    y=alt.Y("Model:N", sort=None, title=None),
).properties(title="How far off is each model?", width=450, height=100)

**What this means**
- A model is only useful if it beats the lazy baseline. Compare the two rows above: the smaller the numbers, the better the guesses.
- The baseline row is the same as on the poster (MAE 10.27, RMSE 14.57), because it uses the same test days.

## Step 6. What does XGBoost rely on?

While it learns, XGBoost keeps track of which clues helped it the most. A longer bar means a more important clue.

In [ ]:
importance = pd.Series(model.feature_importances_, index=clues)              # one number per clue
importance = importance.rename_axis("Clue").reset_index(name="Importance")   # turn it into a table for Altair

alt.Chart(importance).mark_bar().encode(
    x=alt.X("Importance:Q", title="Importance (longer bar = XGBoost used this clue more)"),
    y=alt.Y("Clue:N", sort="-x", title=None),
).properties(title="Which clues does XGBoost use the most?", width=450, height=300)

**What this means**
- Read the chart from the top: the first bars are the clues XGBoost relied on most. Do they make sense?
- Air quality changes slowly from one day to the next, so it is no surprise if today's AQI and the recent average are among the longest bars.

## Step 7. Where does it struggle?

The very bad days matter most for health. Let's see how XGBoost did on the test days when tomorrow's AQI was above 100.

In [ ]:
guess = pd.Series(model.predict(X_test), index=y_test.index)       # XGBoost's guess for every test day

bad_days = y_test > 100                                            # days when tomorrow's AQI was above 100
print("Test days with AQI above 100:", bad_days.sum())
print("Of those, days where XGBoost guessed above 100:", (guess[bad_days] > 100).sum())
print("On those days the real AQI averaged", round(y_test[bad_days].mean()), "and XGBoost's guess averaged", round(guess[bad_days].mean()))

In [ ]:
# One county, June to August 2025: the real AQI against XGBoost's guess
county = "Cuyahoga"                                                # Cleveland. Change this to any county you like.
three_months = test[(test.County == county) & (test.Date >= "2025-06-01") & (test.Date < "2025-09-01")]

compare = pd.DataFrame({"Date": three_months.Date + pd.Timedelta(days=1),   # a guess made today is for tomorrow
                        "Real AQI": three_months.tomorrow,
                        "XGBoost's guess": guess[three_months.index]})
compare = compare.melt("Date", var_name="Line", value_name="AQI")      # one column for the numbers, one for which line

alt.Chart(compare).mark_line().encode(
    x=alt.X("Date:T", title=None), y="AQI:Q", color="Line:N",
).properties(title=f"{county} County, June to August 2025", width=650, height=250)

**What this means**
- XGBoost follows the normal ups and downs, but it does not see the spikes coming: when the real AQI jumps, its guess stays much lower.
- So it is useful on normal days, but it is **not a warning system** for the worst days.

## Summary in plain English

- **Where and when:** the air is worst in the second and third quarters (April to September).
- **Guessing tomorrow:** XGBoost beats the lazy "same as today" guess. Step 5 shows the numbers.
- **Limit:** it works on normal days but misses most of the spikes, so it is not a warning system yet. Step 7 shows the count.
- **Next step:** add next-day weather forecasts and wildfire smoke data to help with the spikes.